<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [2]:
dfs_to_concat = []
for file_path in archivos_anio:
    df_temp = pd.read_csv(file_path)
    df_temp.columns = df_temp.columns.str.lower() # Normalize column names to lowercase
    dfs_to_concat.append(df_temp)

df_anio = pd.concat(dfs_to_concat, ignore_index=True)

# Explore df_codigos to identify and remove inconsistent entry
# As per the notebook state, 'ZWE' is associated with 'Zimbabue' and 'malo'.
# 'malo' is the inconsistent entry that needs to be removed.
df_codigos_cleaned = df_codigos[~( (df_codigos['codigo_iso'] == 'ZWE') & (df_codigos['pais'] == 'malo') )]

# Combine df_anio with df_codigos_cleaned
df = pd.merge(df_anio, df_codigos_cleaned, on='codigo_iso', how='inner')



### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [4]:
print('#### Estructura del DataFrame')
print(f"Filas (observaciones): {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("Nombres de las columnas:", df.columns.tolist())
print("Tipos de datos por columna:")
print(df.info())

print('\n#### Resumen estadístico')
print(df.describe())
print("\nObservaciones sobre 'indice' y 'ranking':\n- 'indice' tiene un rango considerable (min a max), sugiriendo una buena variabilidad en la libertad de prensa. Los valores NaN indican datos faltantes. El ranking también muestra un amplio rango.")
print(f"Valor mínimo de indice: {df['indice'].min()}")
print(f"Valor máximo de indice: {df['indice'].max()}")
print(f"Valor promedio de indice: {df['indice'].mean():.2f}")

print("\nPaíses con valores extremos en 'indice' y 'ranking':")
print("País con el menor índice (mayor libertad de prensa):\n", df.loc[df['indice'].idxmin()])
print("País con el mayor índice (menor libertad de prensa):\n", df.loc[df['indice'].idxmax()])
print("País con el menor ranking (mayor libertad de prensa):\n", df.loc[df['ranking'].idxmin()])
print("País con el mayor ranking (menor libertad de prensa):\n", df.loc[df['ranking'].idxmax()])

print('\n#### Datos faltantes')
missing_values = df.isnull().sum()
print("Valores nulos por columna:\n", missing_values)
missing_proportion = (missing_values / len(df)) * 100
print("\nProporción de observaciones con valores faltantes (%):\n", missing_proportion)
high_missing = missing_proportion[missing_proportion > 30]
print("\nColumnas con más del 30% de datos faltantes:\n", high_missing)

print('\n#### Unicidad y duplicados')
print(f"Países distintos: {df['pais'].nunique()}")
print(f"Años distintos: {df['anio'].nunique()}")
print(f"Filas duplicadas (exactamente iguales): {df.duplicated().sum()}")

print('\n#### Validación cruzada de columnas')
inconsistent_iso_country = df.groupby('codigo_iso')['pais'].nunique()
inconsistent_iso_country = inconsistent_iso_country[inconsistent_iso_country > 1]

if not inconsistent_iso_country.empty:
    print("Códigos ISO asociados a más de un país (inconsistencias):\n")
    for iso_code in inconsistent_iso_country.index:
        print(f"- Código ISO: {iso_code}, Países: {df[df['codigo_iso'] == iso_code]['pais'].unique().tolist()}")
else:
    print("No se encontraron inconsistencias donde un mismo código ISO esté asociado a más de un país.")

#### Estructura del DataFrame
Filas (observaciones): 3060
Columnas: 5
Nombres de las columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']
Tipos de datos por columna:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   anio        3060 non-null   int64  
 2   indice      2664 non-null   float64
 3   ranking     2837 non-null   float64
 4   pais        3060 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB
None

#### Resumen estadístico
              anio        indice        ranking
count  3060.000000   2664.000000    2837.000000
mean   2009.941176    205.782316     477.930913
std       5.786024   2695.525264    6474.935347
min    2001.000000      0.000000       1.000000
25%    2005.000000     15.295000      34.000000
50%    2009.000000     28.000000      70.000000
7




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [6]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
       'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
       'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
       'USA', 'VEN']

df_america = df[df['codigo_iso'].isin(america)].copy()

print("#### a) Usando un ciclo 'for' ####")
# Asegurarse de que el 'anio' esté ordenado y sin duplicados para iterar
for anio in sorted(df_america['anio'].unique()):
    df_anio = df_america[df_america['anio'] == anio]

    # Eliminar filas con NaN en 'indice' para evitar errores en idxmin/idxmax
    df_anio_clean = df_anio.dropna(subset=['indice'])

    if not df_anio_clean.empty:
        # País con menor índice (mayor libertad de prensa)
        min_indice_pais = df_anio_clean.loc[df_anio_clean['indice'].idxmin()]

        # País con mayor índice (menor libertad de prensa)
        max_indice_pais = df_anio_clean.loc[df_anio_clean['indice'].idxmax()]

        print(f"\nAño {anio}:")
        print(f"  Mayor libertad de prensa (menor índice): {min_indice_pais['pais']} (Índice: {min_indice_pais['indice']:.2f})")
        print(f"  Menor libertad de prensa (mayor índice): {max_indice_pais['pais']} (Índice: {max_indice_pais['indice']:.2f})")
    else:
        print(f"\nAño {anio}: No hay datos de índice disponibles para países latinoamericanos.")


print("\n#### b) Usando un enfoque vectorizado con 'groupby' ####")
# Encontrar el índice del mínimo y máximo para cada año
# Drop NaN values from the index series before using .loc
idx_min = df_america.groupby('anio')['indice'].idxmin().dropna()
idx_max = df_america.groupby('anio')['indice'].idxmax().dropna()

# Obtener los DataFrames con los países correspondientes
paises_min_indice = df_america.loc[idx_min]
paises_max_indice = df_america.loc[idx_max]

# Imprimir los resultados
for anio in sorted(df_america['anio'].unique()):
    min_pais_row = paises_min_indice[paises_min_indice['anio'] == anio]
    max_pais_row = paises_max_indice[paises_max_indice['anio'] == anio]

    if not min_pais_row.empty and not max_pais_row.empty:
        min_pais = min_pais_row['pais'].iloc[0]
        min_indice_val = min_pais_row['indice'].iloc[0]
        max_pais = max_pais_row['pais'].iloc[0]
        max_indice_val = max_pais_row['indice'].iloc[0]

        print(f"\nAño {anio}:")
        print(f"  Mayor libertad de prensa (menor índice): {min_pais} (Índice: {min_indice_val:.2f})")
        print(f"  Menor libertad de prensa (mayor índice): {max_pais} (Índice: {max_indice_val:.2f})")
    else:
        print(f"\nAño {anio}: No hay datos de índice disponibles para países latinoamericanos.")

#### a) Usando un ciclo 'for' ####

Año 2001:
  Mayor libertad de prensa (menor índice): Canadá (Índice: 0.80)
  Menor libertad de prensa (mayor índice): Cuba (Índice: 90.30)

Año 2002:
  Mayor libertad de prensa (menor índice): Trinidad y Tobago (Índice: 1.00)
  Menor libertad de prensa (mayor índice): Cuba (Índice: 97.83)

Año 2003:
  Mayor libertad de prensa (menor índice): Trinidad y Tobago (Índice: 2.00)
  Menor libertad de prensa (mayor índice): Argentina (Índice: 35826.00)

Año 2004:
  Mayor libertad de prensa (menor índice): Trinidad y Tobago (Índice: 2.00)
  Menor libertad de prensa (mayor índice): Cuba (Índice: 87.00)

Año 2005:
  Mayor libertad de prensa (menor índice): Bolivia (Índice: 4.50)
  Menor libertad de prensa (mayor índice): Cuba (Índice: 95.00)

Año 2006:
  Mayor libertad de prensa (menor índice): Canadá (Índice: 4.88)
  Menor libertad de prensa (mayor índice): Cuba (Índice: 96.17)

Año 2007:
  Mayor libertad de prensa (menor índice): Canadá (Índice: 3.33)
  Menor

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [7]:
# Construir la tabla dinámica
pivot_table_indice = df.pivot_table(index='pais', columns='anio', values='indice', aggfunc='max', fill_value=0)

print("#### Tabla dinámica de índice máximo por país y año (valores nulos rellenos con 0):####")
print(pivot_table_indice)

# a) ¿Qué país tiene el mayor valor de indice en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
max_indice_overall = pivot_table_indice.max().max()
country_with_max_indice = pivot_table_indice[pivot_table_indice == max_indice_overall].stack().index[0][0]

# Para el menor distinto de cero, primero filtramos los ceros
min_indice_overall = pivot_table_indice[pivot_table_indice > 0].min().min()
country_with_min_indice = pivot_table_indice[pivot_table_indice == min_indice_overall].stack().index[0][0]

print(f"\n#### Respuestas a las preguntas adicionales:####")
print(f"a) País con el mayor índice general: {country_with_max_indice} (Índice: {max_indice_overall:.2f})")
print(f"   País con el menor índice general (distinto de cero): {country_with_min_indice} (Índice: {min_indice_overall:.2f})")

# b) ¿Qué años presentan en promedio los valores de indice más altos? ¿Y los más bajos?
average_indice_per_year = pivot_table_indice.mean(axis=0)
year_highest_avg = average_indice_per_year.idxmax()
year_lowest_avg = average_indice_per_year.idxmin()

print(f"b) Año con el promedio de índice más alto: {year_highest_avg} (Promedio: {average_indice_per_year.max():.2f})")
print(f"   Año con el promedio de índice más bajo: {year_lowest_avg} (Promedio: {average_indice_per_year.min():.2f})")

# c) ¿Qué país muestra mayor variabilidad (diferencia entre su máximo y mínimo indice a lo largo del tiempo)?
# Excluir los 0s para el cálculo de min, a menos que el país solo tenga 0s
variability = pivot_table_indice.apply(lambda row: row[row > 0].max() - row[row > 0].min() if not row[row > 0].empty else 0, axis=1)
country_highest_variability = variability.idxmax()
max_variability_value = variability.max()

print(f"c) País con mayor variabilidad (max-min): {country_highest_variability} (Variabilidad: {max_variability_value:.2f})")

# d) ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?
# Consideramos constante si la variabilidad es 0 (después de filtrar 0s)
constant_indice_countries = variability[variability == 0].index.tolist()

# Filtramos los países que tienen todos sus valores como 0 (estos serán tratados en la pregunta e)
constant_indice_countries = [country for country in constant_indice_countries if not (pivot_table_indice.loc[country] == 0).all()]

if constant_indice_countries:
    print(f"d) Países con índice constante (variabilidad 0, excluyendo solo 0s): {', '.join(constant_indice_countries)}")
else:
    print("d) No se encontraron países con índice constante a lo largo del tiempo (excluyendo solo 0s).")

# e) ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?
countries_all_zeros = pivot_table_indice[(pivot_table_indice == 0).all(axis=1)].index.tolist()
print(f"e) Países sin ningún dato (todos los valores 0): {', '.join(countries_all_zeros)}")
print("   Explicación: Estos países no tenían registros de 'indice' en el DataFrame original para ninguno de los años, o bien todos sus índices eran 0 (lo cual es raro). El 'fill_value=0' en la pivot_table asignó 0 a los valores faltantes.")

#### Tabla dinámica de índice máximo por país y año (valores nulos rellenos con 0):####
anio              2001   2002   2003   2004   2005   2006   2007   2008  \
pais                                                                      
Afghanistán       35.5  40.17  28.25  39.17  44.25  56.50  59.25  54.25   
Albania            0.0   6.50  11.50  14.17  18.00  25.50  16.00  21.75   
Alemania           1.5   1.33   2.00   4.00   5.50   5.75   4.50   3.50   
Algeria           31.0  33.00  43.50  40.33  40.00  40.50  31.33  49.56   
Andorra            0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00   
...                ...    ...    ...    ...    ...    ...    ...    ...   
Vietnam           81.3  89.17  86.88  73.25  67.25  79.25  86.17  81.67   
West Bank y Gaza   0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00   
Yemen             34.8  41.83  48.00  46.25  54.00  56.67  59.00  83.38   
Zambia            26.8  23.25  29.75  23.00  22.50  21.50  15.50  26.75   
Zimbabue    